# Lekcija 35: Treniranje CNN-a

CNN iz lekcije 34 treniran je i evaluiran na podacima iz iste, prilično bogate raspodjele. U stvarnom treniranju vreba mnogo izraženiji način neuspjeha: uz premalo podataka i prevelik kapacitet modela mreža može savršeno zapamtiti skup za treniranje, a ne naučiti ništa što generalizira. Lekcija 32 pokazala je da veći kapacitet znači da model *može* predstavljati više; ova lekcija pokazuje drugu stranu — upravo taj višak kapaciteta, uz premalo podataka koji ga ograničavaju, omogućuje mreži da pamti umjesto da generalizira. Ova lekcija konkretno prikazuje taj neuspjeh, a zatim ga ispravlja na tri načina: **augmentacijom podataka** (transformacije iz lekcije 8, s novom namjenom), **regularizacijom** (smanjivanje težina) i **dropoutom**.

In [ ]:
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Namjerno težak, mali skup podataka

Isti zadatak razlikovanja plusa i kruga kao u lekciji 34, ali sada sa samo **12 slika za treniranje** i pikselnim šumom dodanim svakoj slici, dok validacijski skup ostaje velik (150 slika) kako bi njegova točnost bila pouzdana procjena.

In [ ]:
def make_image(shape_type, cx, cy, size=16, rng=None):
    img = np.zeros((size, size), dtype=np.float32)
    if shape_type == 'plus':
        img[cy - 1:cy + 2, cx - 3:cx + 4] = 1.0
        img[cy - 3:cy + 4, cx - 1:cx + 2] = 1.0
    else:
        yy, xx = np.mgrid[0:size, 0:size]
        img[((xx - cx) ** 2 + (yy - cy) ** 2) <= 9] = 1.0
    if rng is not None:
        img = np.clip(img + rng.normal(0, 0.4, img.shape), 0, 1).astype(np.float32)
    return img

def make_dataset(rng_local, n, position_range):
    imgs, labels = [], []
    for _ in range(n):
        shape_type = rng_local.choice(['plus', 'circle'])
        cx, cy = rng_local.integers(*position_range), rng_local.integers(*position_range)
        imgs.append(make_image(shape_type, cx, cy, rng=rng_local))
        labels.append(0.0 if shape_type == 'plus' else 1.0)
    return np.array(imgs, dtype=np.float32), np.array(labels, dtype=np.float32)

data_rng = np.random.default_rng(2)
X_train, y_train = make_dataset(data_rng, 12, (3, 13))
X_val, y_val = make_dataset(data_rng, 150, (3, 13))

fig, axes = plt.subplots(1, 6, figsize=(11, 2))
for ax, im in zip(axes, X_train[:6]):
    ax.imshow(im, cmap='gray')
    ax.axis('off')
fig.suptitle('The entire training set is only 12 noisy images like these', y=1.05)
plt.show()

## Promatranje prenaučenosti

Trenirajte razmjerno velik CNN (arhitekturu iz lekcije 34) na samo tih 12 slika i u svakoj epohi pratite gubitak na skupu za treniranje i *validacijski* gubitak (na izdvojenih 150 slika).

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 5, padding=2), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 5, padding=2), nn.ReLU(),
            nn.AdaptiveMaxPool2d(1),
        )
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        return self.fc(self.conv(x).flatten(1)).squeeze(-1)

def train_tracked(model_cls, Xtr, ytr, Xval, yval, epochs=400, lr=0.01, weight_decay=0.0, seed=0):
    torch.manual_seed(seed)  # seed BEFORE constructing the model, so init is actually reproducible
    model = model_cls()
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    train_losses, val_losses = [], []
    for _ in range(epochs):
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(Xtr), ytr)
        loss.backward()
        opt.step()
        with torch.no_grad():
            train_losses.append(loss.item())
            val_losses.append(F.binary_cross_entropy_with_logits(model(Xval), yval).item())
    with torch.no_grad():
        train_acc = ((model(Xtr) > 0).float() == ytr).float().mean().item()
        val_acc = ((model(Xval) > 0).float() == yval).float().mean().item()
    return train_losses, val_losses, train_acc, val_acc

Xtr_t = torch.tensor(X_train).unsqueeze(1); ytr_t = torch.tensor(y_train)
Xval_t = torch.tensor(X_val).unsqueeze(1); yval_t = torch.tensor(y_val)

train_losses, val_losses, train_acc, val_acc = train_tracked(CNN, Xtr_t, ytr_t, Xval_t, yval_t)

print(f'final train accuracy: {train_acc:.1%}')
print(f'final val accuracy:   {val_acc:.1%}')
print(f'val loss minimum was {min(val_losses):.3f} at epoch {np.argmin(val_losses)} '
      f'(out of {len(val_losses)}); it ended at {val_losses[-1]:.3f}')

plt.plot(train_losses, label='train loss')
plt.plot(val_losses, label='val loss')
plt.axvline(np.argmin(val_losses), color='gray', linestyle='--', linewidth=1, label='best val loss')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(fontsize=8)
plt.title('The classic overfitting curve')
plt.show()

Gubitak na skupu za treniranje postojano pada prema nuli &mdash; mreža savršeno pamti svih 12 slika, uključujući šum. Validacijski gubitak u međuvremenu doseže minimum tijekom treniranja, a zatim ponovno raste: nakon te točke svaka dodatna epoha čini model *još* sigurnijim u pogrešna predviđanja za neviđene podatke. Završna validacijska točnost ostaje znatno ispod savršenog rezultata na skupu za treniranje, unatoč njegovoj potpunoj prilagodbi.

## Rješenje 1: augmentacija podataka

Ako nema dovoljno stvarnih podataka, proizvedite više iz postojećih. Primijenite nasumične transformacije iz lekcije 8 (zrcaljenja, male rotacije) na svaku sliku za treniranje &mdash; oznaka se ne mijenja, ali pikseli se mijenjaju, pa mreža vidi mnogo raznovrsnije primjere „kako plus ili krug mogu izgledati” umjesto da pamti 12 točnih slika.

In [ ]:
def augment(img, rng_local):
    if rng_local.random() < 0.5:
        img = np.fliplr(img).copy()
    if rng_local.random() < 0.5:
        img = np.flipud(img).copy()
    angle = rng_local.uniform(-10, 10)
    M = cv2.getRotationMatrix2D((8, 8), angle, 1.0)
    img = cv2.warpAffine(img, M, (16, 16))
    return img.astype(np.float32)

aug_rng = np.random.default_rng(5)
X_aug, y_aug = [], []
for _ in range(20):  # 20 augmented copies of each of the 12 original images
    for img, label in zip(X_train, y_train):
        X_aug.append(augment(img, aug_rng))
        y_aug.append(label)
X_aug, y_aug = np.array(X_aug, dtype=np.float32), np.array(y_aug, dtype=np.float32)

fig, axes = plt.subplots(1, 6, figsize=(11, 2))
for ax, im in zip(axes, X_aug[:6]):
    ax.imshow(im, cmap='gray')
    ax.axis('off')
fig.suptitle(f'6 of {len(X_aug)} augmented copies, all still "the same 12 base images"', y=1.05)
plt.show()

Xaug_t = torch.tensor(X_aug).unsqueeze(1); yaug_t = torch.tensor(y_aug)
_, _, aug_train_acc, aug_val_acc = train_tracked(CNN, Xaug_t, yaug_t, Xval_t, yval_t, epochs=150)
print(f'with augmentation: train acc = {aug_train_acc:.1%}, val acc = {aug_val_acc:.1%}  (was {val_acc:.1%})')

## Rješenje 2: smanjivanje težina

**Smanjivanje težina (weight decay)** izravno u gubitak dodaje kaznu proporcionalnu kvadratu veličine težina (ekvivalentno, pri svakom ažuriranju svaku težinu malo približava nuli). Velike, precizno podešene težine upravo su ono što mreža treba za pamćenje 12 konkretnih slika sa šumom; kažnjavanje veličine težina čini takvo pamćenje skupljim u odnosu na pronalaženje jednostavnije, glađe funkcije &mdash; bez dodavanja ijednog primjera za treniranje.

In [ ]:
_, _, wd_train_acc, wd_val_acc = train_tracked(CNN, Xtr_t, ytr_t, Xval_t, yval_t, weight_decay=0.05)
print(f'with weight_decay=0.05: train acc = {wd_train_acc:.1%}, val acc = {wd_val_acc:.1%}  (was {val_acc:.1%})')

print()
print(f'{"approach":>20} {"train acc":>12} {"val acc":>12}')
print(f'{"no fix":>20} {train_acc:>12.1%} {val_acc:>12.1%}')
print(f'{"augmentation":>20} {aug_train_acc:>12.1%} {aug_val_acc:>12.1%}')
print(f'{"weight decay":>20} {wd_train_acc:>12.1%} {wd_val_acc:>12.1%}')

Oba rješenja vraćaju velik dio izgubljene validacijske točnosti, iz dvaju različitih smjerova: augmentacija modelu daje više (sintetičkih) podataka o kojima treba biti u pravu; smanjivanje težina čini model manje sklonim pretjeranoj prilagodbi malom skupu podataka. U praksi se oba obično koriste zajedno, uz druge regularizatore poput **dropouta**, koji slijedi, i **normalizacije serije** (lekcija 37), koja usput djeluje i kao blag regularizator.

## Rješenje 3: dropout

**Dropout** (<a href="../references.html#srivastava-2014">Srivastava i sur., 2014.</a><span class="landmark-paper">&#9733;</span>) pri svakom koraku treniranja nasumično postavlja udio `p` aktivacija sloja na nulu, prisiljavajući preostale jedinice da ne ovise o stalnoj prisutnosti neke određene druge jedinice. Kako bi ukupno mjerilo izlaza sloja ostalo isto bez obzira na aktivnost dropouta, preostale aktivacije skaliraju se s `1 / (1 - p)` — to je „invertirani dropout”, ono što sloj `Dropout` svakog razvojnog okvira zapravo implementira. Pri evaluaciji dropout ne radi ništa: izvršava se cijeli, neizmijenjeni sloj. Zato je važan `model.eval()` (koji se već koristi kroz ovu lekciju, i za smanjivanje težina i za augmentaciju) — njegovim izostavljanjem dropout bi ostao nasumično aktivan pri testiranju.

In [ ]:
def manual_dropout(x, p, rng_gen):
    keep_prob = 1 - p
    mask = (torch.rand(x.shape, generator=rng_gen) < keep_prob).float()
    return x * mask / keep_prob  # rescale so E[output] == input

x_demo = torch.randn(2000, 10)
torch.manual_seed(0)
out_torch = F.dropout(x_demo, p=0.3, training=True)
out_manual = manual_dropout(x_demo, 0.3, torch.Generator().manual_seed(0))

print(f'fraction zeroed, torch:  {(out_torch == 0).float().mean().item():.3f}  (target p = 0.3)')
print(f'fraction zeroed, manual: {(out_manual == 0).float().mean().item():.3f}')
print(f'mean before dropout: {x_demo.mean().item():.4f}')
print(f'mean after dropout (torch):  {out_torch.mean().item():.4f}  (rescaling keeps this close to the input mean)')
print(f'mean after dropout (manual): {out_manual.mean().item():.4f}')

eval_mode_out = F.dropout(x_demo, p=0.3, training=False)
print(f'eval-mode dropout is a no-op: {torch.equal(eval_mode_out, x_demo)}')

Sada ga primijenimo na problem prenaučenosti iz ove lekcije. Dropout za djelovanje treba *određenu* redundantnost — postavljanje polovice 32-dimenzijskog vektora značajki na nulu neposredno prije izlaza s jednom jedinicom ostavlja malo prostora za poboljšanje. Zato dodajemo širi skriveni sloj (32 → 64 → 1) i postavljamo dropout na sloj sa 64 jedinice. Uz samo 12 slika za treniranje rezultati znatno variraju između nasumičnih inicijalizacija, pa uspoređujemo prosječnu validacijsku točnost kroz više inicijalizacija umjesto da vjerujemo jednom pokretanju.

In [ ]:
class CNNDropout(nn.Module):
    def __init__(self, dropout_p=0.0):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 5, padding=2), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 5, padding=2), nn.ReLU(),
            nn.AdaptiveMaxPool2d(1),
        )
        self.fc1 = nn.Linear(32, 64)
        self.dropout = nn.Dropout(dropout_p)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        feat = torch.relu(self.fc1(self.conv(x).flatten(1)))
        return self.fc2(self.dropout(feat)).squeeze(-1)

def val_acc_for(dropout_p, seed, epochs=150):
    torch.manual_seed(seed)
    model = CNNDropout(dropout_p)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(Xtr_t), ytr_t)
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        return ((model(Xval_t) > 0).float() == yval_t).float().mean().item()

no_drop_accs = [val_acc_for(0.0, seed) for seed in range(8)]
drop_accs = [val_acc_for(0.5, seed) for seed in range(8)]

print(f'no dropout:     mean val acc = {np.mean(no_drop_accs):.1%}  (+/- {np.std(no_drop_accs):.1%}, 8 seeds)')
print(f'dropout(0.5):   mean val acc = {np.mean(drop_accs):.1%}  (+/- {np.std(drop_accs):.1%}, 8 seeds)')

U ovako malom demonstracijskom postavu (samo 12 slika za treniranje) još nema mnogo redundantnosti koju dropout može iskoristiti. Iako je ovdje prikazano samo skromno poboljšanje, korist dropouta dobro je potvrđena na stvarnim skupovima podataka i mrežama (stotine redundantnih jedinica, tisuće primjera). U praksi se dropout gotovo uvijek kombinira s ostalim regularizatorima s ove stranice, umjesto da se koristi sam.

## Povezivanje svega: stvarne fotografije, stvarna prenaučenost

Svako dosadašnje rješenje testirano je zasebno na sintetičkom demonstracijskom problemu s 12 slika, odabranom baš zato da način neuspjeha bude nedvojben. Ovaj završni odjeljak ponavlja eksperiment na stvarnim fotografijama iz **CIFAR-10** (<a href="../references.html#krizhevsky-2009-cifar">Krizhevsky, 2009.</a>): malom skupu za treniranje s namjerno malo podataka o mačkama i kamionima, uz zajedničku primjenu augmentacije, smanjivanja težina i dropouta. Stvarne fotografije sadrže osvjetljenje, vizualni nered i stvarnu vizualnu dvosmislenost između klasa.


In [ ]:
import pickle
import tarfile
import urllib.request
from pathlib import Path

CIFAR_URL = 'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz'
CACHE_ROOT = Path.home() / '.cache' / 'cvintro'
CACHE_DIR = CACHE_ROOT / 'cifar-10-batches-py'

def ensure_cifar10():
    if CACHE_DIR.exists():
        return
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    archive_path = CACHE_ROOT / 'cifar-10-python.tar.gz'
    if not archive_path.exists():
        print('Downloading CIFAR-10 (~163 MB, one-time, cached under ~/.cache/cvintro)...')
        urllib.request.urlretrieve(CIFAR_URL, archive_path)
    print('Extracting...')
    with tarfile.open(archive_path) as tar:
        tar.extractall(CACHE_ROOT)

def load_cifar_batch(path):
    with open(path, 'rb') as f:
        d = pickle.load(f, encoding='bytes')
    imgs = d[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype(np.float32) / 255.0
    labels = np.array(d[b'labels'], dtype=np.int64)
    return imgs, labels

CIFAR_LABELS = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

ensure_cifar10()
cifar_train_imgs, cifar_train_labels = [], []
for i in range(1, 6):
    imgs, labels = load_cifar_batch(CACHE_DIR / f'data_batch_{i}')
    cifar_train_imgs.append(imgs)
    cifar_train_labels.append(labels)
cifar_train_imgs = np.concatenate(cifar_train_imgs)
cifar_train_labels = np.concatenate(cifar_train_labels)
cifar_test_imgs, cifar_test_labels = load_cifar_batch(CACHE_DIR / 'test_batch')

def take_class(imgs, labels, name, n, rng_local):
    idx = np.where(labels == CIFAR_LABELS.index(name))[0]
    idx = rng_local.permutation(idx)[:n]
    return imgs[idx].copy()

cifar_rng = np.random.default_rng(4)
N_CIFAR_TRAIN = 40  # per class -- deliberately small, real training data is expensive
X_cifar_train = np.concatenate([
    take_class(cifar_train_imgs, cifar_train_labels, 'cat', N_CIFAR_TRAIN, cifar_rng),
    take_class(cifar_train_imgs, cifar_train_labels, 'truck', N_CIFAR_TRAIN, cifar_rng),
])
y_cifar_train = np.array([0.0] * N_CIFAR_TRAIN + [1.0] * N_CIFAR_TRAIN, dtype=np.float32)
X_cifar_val = np.concatenate([
    take_class(cifar_test_imgs, cifar_test_labels, 'cat', 150, cifar_rng),
    take_class(cifar_test_imgs, cifar_test_labels, 'truck', 150, cifar_rng),
])
y_cifar_val = np.array([0.0] * 150 + [1.0] * 150, dtype=np.float32)

fig, axes = plt.subplots(1, 6, figsize=(11, 2))
for ax, im in zip(axes, X_cifar_train[:6]):
    ax.imshow(im)
    ax.axis('off')
fig.suptitle(f'6 of {len(X_cifar_train)} real training photos (cat vs. truck)', y=1.05)
plt.show()

Isti postupak kao prije, prilagođen trokanalnom ulazu: obični CNN bez regularizacije uspoređujemo s istom arhitekturom uz augmentaciju, smanjivanje težina i dropout primijenjene zajedno u jednom treniranju. Svaku konfiguraciju usrednjavamo preko 3 inicijalizacije jer je 80 slika za treniranje još uvijek dovoljno malo da pojedinačna pokretanja variraju.

In [ ]:
class CNNRGB(nn.Module):
    def __init__(self, dropout_p=0.0):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 5, padding=2), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 5, padding=2), nn.ReLU(),
            nn.AdaptiveMaxPool2d(1),
        )
        self.fc1 = nn.Linear(32, 64)
        self.dropout = nn.Dropout(dropout_p)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        feat = torch.relu(self.fc1(self.conv(x).flatten(1)))
        return self.fc2(self.dropout(feat)).squeeze(-1)

def augment_photo(img, rng_local):
    # unlike `augment` above, no vertical flip: upside-down cats and trucks aren't realistic
    if rng_local.random() < 0.5:
        img = np.fliplr(img).copy()
    angle = rng_local.uniform(-10, 10)
    M = cv2.getRotationMatrix2D((16, 16), angle, 1.0)
    img = cv2.warpAffine(img, M, (32, 32))
    return img.astype(np.float32)

Xcv_t = torch.tensor(X_cifar_val).permute(0, 3, 1, 2); ycv_t = torch.tensor(y_cifar_val)

def train_cifar(use_aug, weight_decay, dropout_p, seed, epochs=300, lr=0.001):
    torch.manual_seed(seed)
    if use_aug:
        local_aug_rng = np.random.default_rng(seed + 100)
        Xa, ya = [], []
        for _ in range(10):  # 10 augmented copies of each of the 80 base images
            for img, label in zip(X_cifar_train, y_cifar_train):
                Xa.append(augment_photo(img, local_aug_rng))
                ya.append(label)
        Xtr_np, ytr_np = np.array(Xa, dtype=np.float32), np.array(ya, dtype=np.float32)
    else:
        Xtr_np, ytr_np = X_cifar_train, y_cifar_train
    Xct_t = torch.tensor(Xtr_np).permute(0, 3, 1, 2); yct_t = torch.tensor(ytr_np)
    model = CNNRGB(dropout_p=dropout_p)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(Xct_t), yct_t)
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        train_acc = ((model(Xct_t) > 0).float() == yct_t).float().mean().item()
        val_acc = ((model(Xcv_t) > 0).float() == ycv_t).float().mean().item()
    return train_acc, val_acc

cifar_configs = [
    ('no fix',       dict(use_aug=False, weight_decay=0.0,  dropout_p=0.0)),
    ('augmentation', dict(use_aug=True,  weight_decay=0.0,  dropout_p=0.0)),
    ('weight decay', dict(use_aug=False, weight_decay=0.01, dropout_p=0.0)),
    ('dropout',      dict(use_aug=False, weight_decay=0.0,  dropout_p=0.3)),
    ('all combined', dict(use_aug=True,  weight_decay=0.01, dropout_p=0.3)),
]

print(f'{"approach":>15} {"train acc":>12} {"val acc":>12}')
cifar_results = {}
for name, cfg in cifar_configs:
    tr_accs, val_accs = [], []
    for seed in range(3):
        tr, va = train_cifar(seed=seed, **cfg)
        tr_accs.append(tr)
        val_accs.append(va)
    cifar_results[name] = (np.mean(tr_accs), np.mean(val_accs))
    print(f'{name:>15} {np.mean(tr_accs):>12.1%} {np.mean(val_accs):>12.1%}')

Nijedno pojedinačno rješenje ovdje nije jasan pobjednik. Samo smanjivanje težina zapravo završava *ispod* početnog modela bez regularizacije — uz `weight_decay=0.01` točnost na skupu za treniranje više ne doseže ni 100 %, što znači da je kazna dovoljno snažna da se suprotstavlja cilju bez pronalaženja boljeg rješenja. To je koristan rezultat koji poziva na oprez: ranije korištena vrijednost `weight_decay=0.05` bila je podešena za mnogo manju mrežu na 12 sitnih sintetičkih slika, a jačina regularizacije koja dobro radi u jednom postavu ne prenosi se automatski na drugu arhitekturu ili stvarne fotografske podatke.

Augmentacija i dropout pojedinačno vraćaju skroman dio validacijske točnosti.

„Sve zajedno” nadmašuje svako pojedinačno rješenje i početni model bez regularizacije, potvrđujući istu pouku iz sintetičkog skupa: ova tri regularizatora međusobno se nadopunjuju, a ne zamjenjuju. Dobitak u odnosu na model bez regularizacije stvaran je, ali skroman (ovdje nekoliko postotnih bodova), što je i iskrena pouka: regularizacija sužava razliku između uspješnosti na skupu za treniranje i validacijskom skupu, ali ne stvara podatke ili kapacitet modela koji ne postoje.

Daljnje smanjivanje te razlike na stvarnim slikama zahtijeva ono što uvode kasnije lekcije — više podataka za treniranje, bolje arhitekture (lekcija 37) i naposljetku **prijenos učenja** (lekcija 38): početak s mrežom već treniranom na milijunima slika, umjesto pamćenja od početka na tek nekoliko njih.

### Zadaci

1. Isprobajte vrijednosti `weight_decay` od `0.001`, `0.05` i `1.0`. Postoji li točka na kojoj počinje *štetiti* točnosti na skupu za treniranje, uz (naposljetku) validacijsku točnost? Što pretjerano veliko smanjivanje težina radi kapacitetu modela da se uopće prilagodi bilo čemu?
2. Mijenjajte `dropout_p` kroz `[0.0, 0.2, 0.4, 0.6, 0.8]` u `val_acc_for`, usrednjavajući preko istih 8 inicijalizacija za svaku vrijednost. Postoji li očito najbolja vrijednost ili prosjek ostaje unutar jedne standardne devijacije kroz većinu raspona s obzirom na tako malo podataka?
3. Povećajte množitelj augmentacije s 20 na 100 kopija po osnovnoj slici. Nastavlja li se validacijska točnost poboljšavati ili doseže plato — a ako ga doseže, što to govori o temeljnom ograničenju augmentacije skupa koji od početka sadrži samo 12 *izvornih* primjera?